# 04 — Diverse baselines on identical folds

boosting、bagging、線形、ニューラルネットを同じ特徴量・同じfoldで比較します。
単体AUCだけでなく予測誤差の違いを作ることが、後続アンサンブルの目的です。

In [ ]:
from pathlib import Path
import sys
import time

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from features import get_base_features
from train import MODEL_ORDER, make_model, prepare_catboost_frame, run_cv

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
FOLD_COLUMN = Baseline.FOLD_COLUMN
USE_GPU = False

train = pd.read_csv(ROOT / "input" / "train.csv")
test = pd.read_csv(ROOT / "input" / "test.csv")
train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0})
folds = pd.read_csv(ROOT / "output" / "stkfolds.csv")
train = train.merge(folds, on=ID_COLUMN, how="left", validate="one_to_one")
assert train[FOLD_COLUMN].notna().all()

BASE_FEATURES, BASE_NUM, BASE_CAT = get_base_features(
    train, TARGET, ID_COLUMN, min_nunique=Baseline.MIN_NUMERIC_UNIQUE
)
print("train:", train.shape, "test:", test.shape)
print("numeric:", BASE_NUM)
print("categorical:", BASE_CAT)

## 共通条件

- 5モデルとも同じ `stkfolds.csv` を使用
- sklearnモデルの欠損補完・one-hot・標準化はfold内でfit
- CatBoostだけはカテゴリ列を文字列へ変換し、native categorical処理を使用
- OOF/test予測をモデルごとに保存

In [ ]:
results = {}
rows = []

for model_name in MODEL_ORDER:
    print("=" * 60)
    print("Running:", model_name)
    started = time.time()
    model = make_model(
        model_name,
        BASE_NUM,
        BASE_CAT,
        seed=Baseline.SEED,
        use_gpu=USE_GPU,
    )

    if model_name == "CatBoost":
        train_input = train.copy()
        test_input = test.copy()
        train_input[BASE_FEATURES] = prepare_catboost_frame(
            train, BASE_FEATURES, BASE_CAT
        )
        test_input[BASE_FEATURES] = prepare_catboost_frame(
            test, BASE_FEATURES, BASE_CAT
        )
    else:
        train_input, test_input = train, test

    prefix = model_name.lower().replace(" ", "_")
    result = run_cv(
        model=model,
        train=train_input,
        test=test_input,
        features=BASE_FEATURES,
        target=TARGET,
        id_column=ID_COLUMN,
        fold_column=FOLD_COLUMN,
        label=model_name,
        save_prefix=prefix,
        output_dir=ROOT / "artifacts",
    )
    elapsed = time.time() - started
    rows.append(
        {
            "model": model_name,
            "oof_auc": result["oof_auc"],
            "fold_auc_mean": result["fold_df"]["auc"].mean(),
            "fold_auc_std": result["fold_df"]["auc"].std(ddof=0),
            "time_sec": elapsed,
        }
    )
    results[model_name] = result

comparison = pd.DataFrame(rows).sort_values("oof_auc", ascending=False)
display(comparison.reset_index(drop=True))

## 比較の読み方

AUC差が小さい場合、順位だけでは判断しません。foldごとの差、標準偏差、実行時間、
そしてNotebook 05で確認するOOF相関を合わせて見ます。少し弱いモデルでも、
強いモデルと異なる誤り方ならアンサンブルに価値があります。